# Day FINAL DAY of ML 30-Days Challenge

# Ultimate Project for Final Day

**Customer Churn Prediction & Model Optimization**<br><br>
**The Goal**: You are a data scientist at a telecom company. Your task is to build a machine learning model that can predict which customers are likely to "churn" (cancel their subscription). This is a high-value project because it's much cheaper to retain an existing customer than to acquire a new one.

In [43]:
!pip install xgboost

In [65]:
import os, pandas as pd, numpy as np

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.decomposition import PCA
from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support, roc_auc_score,
    classification_report, confusion_matrix
)
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

try:
    from xgboost import XGBClassifier
    HAS_XGB = True
    print("XGBoost imported successfully.")
except ImportError:
    HAS_XGB = False
    print("XGBoost not found.")

XGBoost imported successfully.


###1) Load

In [45]:
path = "/content/Churn.csv"

df = pd.read_csv(path)

print("Total rows : ",len(df),"\n\n")

print("Count of valid values per column:")
for i in df.columns:
  a=len(df[i])-len([i for i in df[i] if i in [' ',np.nan,None]])
  print(i," : ",a,(" <- Missing elements" if a!=len(df) else ""))

Total rows :  7043 


Count of valid values per column:
customerID  :  7043 
gender  :  7043 
SeniorCitizen  :  7043 
Partner  :  7043 
Dependents  :  7043 
tenure  :  7043 
PhoneService  :  7043 
MultipleLines  :  7043 
InternetService  :  7043 
OnlineSecurity  :  7043 
OnlineBackup  :  7043 
DeviceProtection  :  7043 
TechSupport  :  7043 
StreamingTV  :  7043 
StreamingMovies  :  7043 
Contract  :  7043 
PaperlessBilling  :  7043 
PaymentMethod  :  7043 
MonthlyCharges  :  7043 
TotalCharges  :  7032  <- Missing elements
Churn  :  7043 


###2) Basic cleaning

In [46]:
# Known issue: TotalCharges contains blanks -> coerce to numeric
if 'TotalCharges' in df.columns:
    df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

###3) Target and features

In [47]:
target_col = 'Churn'
y = (df[target_col].map({'Yes':1,'No':0}) if df[target_col].dtype=='object' else df[target_col]).astype(int)
X = df.drop(columns=[target_col, 'customerID'] if 'customerID' in df.columns else [target_col])

###4) Column types (must divide for future encoding)

In [48]:
cat_cols = [c for c in X.columns if X[c].dtype == 'object']
num_cols = [c for c in X.columns if c not in cat_cols]

###5) Preprocessing pipelines

In [49]:
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler(with_mean=True, with_std=True))
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, num_cols),
        ('cat', categorical_transformer, cat_cols)
    ]
)

# PCA (will use in LR pipeline; tree models do not benefit but still use imputing/one-hot)
pca = PCA(n_components=0.95, svd_solver='full')

###6) Models via clean pipelines

In [50]:
log_reg = LogisticRegression(max_iter=200, solver='lbfgs')
rf = RandomForestClassifier(random_state=42)
models = {
    'LogisticRegression': Pipeline([('preprocess', preprocessor), ('pca', pca), ('clf', log_reg)]),
    'RandomForest': Pipeline([('preprocess', preprocessor), ('clf', rf)])
}
if HAS_XGB:
    xgb = XGBClassifier(
        objective='binary:logistic',
        n_estimators=300,
        learning_rate=0.1,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.8,
        eval_metric='logloss',
        random_state=42,
        tree_method='hist'
    )
    models['XGBoost'] = Pipeline([('preprocess', preprocessor), ('clf', xgb)])

###7) Split (y-Stratified)

In [51]:
X_train, X_test, y_train, y_test = train_test_split( X, y, test_size=0.10, random_state=42, stratify=y )

###8) Cross-validated evaluation

In [52]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_rows = []
for name, pipe in models.items():
    scores = cross_validate(
        pipe, X_train, y_train, cv=cv,
        scoring={'accuracy':'accuracy', 'roc_auc':'roc_auc', 'f1':'f1', 'precision':'precision', 'recall':'recall'},
        return_train_score=False
    )
    cv_rows.append({
        'model': name,
        'cv_accuracy_mean': np.mean(scores['test_accuracy']),
        'cv_roc_auc_mean': np.mean(scores['test_roc_auc']),
        'cv_f1_mean': np.mean(scores['test_f1']),
        'cv_precision_mean': np.mean(scores['test_precision']),
        'cv_recall_mean': np.mean(scores['test_recall'])
    })

    print(scores)
cv_df = pd.DataFrame(cv_rows).sort_values('cv_roc_auc_mean', ascending=False)
cv_df.to_csv('cv_results.csv', index=False)

pd.read_csv('cv_results.csv')

{'fit_time': array([0.1563983 , 0.18439794, 0.17478943, 0.19452095, 0.19840312]), 'score_time': array([0.12044668, 0.21871758, 0.19855213, 0.10282087, 0.10489726]), 'test_accuracy': array([0.79574132, 0.79968454, 0.79810726, 0.80189424, 0.8050513 ]), 'test_roc_auc': array([0.84765705, 0.82066442, 0.83383427, 0.84705546, 0.84639852]), 'test_f1': array([0.57190083, 0.5968254 , 0.57333333, 0.59186992, 0.59038143]), 'test_precision': array([0.64312268, 0.64163823, 0.6539924 , 0.65232975, 0.66666667]), 'test_recall': array([0.51488095, 0.5578635 , 0.51038576, 0.54166667, 0.5297619 ])}
{'fit_time': array([0.71932602, 0.63414884, 0.65710664, 0.67262745, 0.91983962]), 'score_time': array([0.08311892, 0.08565116, 0.08633661, 0.0937655 , 0.15211248]), 'test_accuracy': array([0.78864353, 0.79416404, 0.77760252, 0.77979479, 0.78610892]), 'test_roc_auc': array([0.83141254, 0.80888901, 0.81581975, 0.82905926, 0.82171308]), 'test_f1': array([0.5472973 , 0.55986509, 0.52525253, 0.53266332, 0.53356282]

,model,cv_accuracy_mean,cv_roc_auc_mean,cv_f1_mean,cv_precision_mean,cv_recall_mean
0,LogisticRegression,0.800096,0.839122,0.584862,0.651550,0.530912
1,XGBoost,0.784633,0.821453,0.561209,0.610763,0.519625
2,RandomForest,0.785263,0.821379,0.539728,0.626020,0.474431


###9) Select best model by ROC AUC, fit, and evaluate on holdout

- Holdout_accuracy :- how well the model will generalize to truly unseen data

In [53]:
best_model_name = cv_df.iloc[0]['model']
best_pipe = models[best_model_name]
best_pipe.fit(X_train, y_train)

proba = best_pipe.predict_proba(X_test)[:, 1]
y_pred = (proba >= 0.5).astype(int)

acc = accuracy_score(y_test, y_pred)
prec, rec, f1, _ = precision_recall_fscore_support(y_test, y_pred, average='binary')
roc = roc_auc_score(y_test, proba)
cm = confusion_matrix(y_test, y_pred)

pd.DataFrame([{
    'best_model': best_model_name,
    'holdout_accuracy': acc,
    'holdout_precision': prec,
    'holdout_recall': rec,
    'holdout_f1': f1,
    'holdout_roc_auc': roc,
    'tn': cm[0,0], 'fp': cm[0,1], 'fn': cm[1,0], 'tp': cm[1,1]
}]).to_csv('holdout_metrics_before_tuning.csv', index=False)

pd.read_csv('holdout_metrics_before_tuning.csv')

,best_model,holdout_accuracy,holdout_precision,holdout_recall,holdout_f1,holdout_roc_auc,tn,fp,fn,tp
0,LogisticRegression,0.808511,0.685714,0.513369,0.587156,0.843949,474,44,91,96


###10) Hyperparameter tuning (GridSearchCV on the best pipeline)

In [54]:
param_grid = {}
if best_model_name == 'LogisticRegression':
    param_grid = {
        'pca__n_components': [0.80, 0.90, 0.95, None],
        'clf__C': [0.1, 1.0, 3.0, 10.0],
        'clf__penalty': ['l2'],
        'clf__solver': ['lbfgs']
    }
elif best_model_name == 'RandomForest':
    param_grid = {
        'clf__n_estimators': [200, 400, 600],
        'clf__max_depth': [None, 8, 12, 16],
        'clf__min_samples_split': [2, 5, 10],
        'clf__min_samples_leaf': [1, 2, 4],
        'clf__max_features': ['sqrt', 'log2', None]
    }
elif best_model_name == 'XGBoost' and HAS_XGB:
    param_grid = {
        'clf__n_estimators': [300, 500, 700],
        'clf__max_depth': [4, 6, 8],
        'clf__learning_rate': [0.03, 0.1, 0.2],
        'clf__subsample': [0.7, 0.9, 1.0],
        'clf__colsample_bytree': [0.7, 0.9, 1.0]
    }

best_params = {}
if param_grid:
    grid = GridSearchCV(best_pipe, param_grid, scoring='roc_auc', cv=cv, n_jobs=-1, refit=True)
    grid.fit(X_train, y_train)
    tuned_best_pipe = grid.best_estimator_
    best_params = grid.best_params_
else:
    tuned_best_pipe = best_pipe

param_grid

{'pca__n_components': [0.8, 0.9, 0.95, None],
 'clf__C': [0.1, 1.0, 3.0, 10.0],
 'clf__penalty': ['l2'],
 'clf__solver': ['lbfgs']}

###11) Evaluate tuned model

In [55]:
proba_tuned = tuned_best_pipe.predict_proba(X_test)[:, 1]
y_pred_tuned = (proba_tuned >= 0.5).astype(int)

acc_t = accuracy_score(y_test, y_pred_tuned)
prec_t, rec_t, f1_t, _ = precision_recall_fscore_support(y_test, y_pred_tuned, average='binary')
roc_t = roc_auc_score(y_test, proba_tuned)
cm_t = confusion_matrix(y_test, y_pred_tuned)

row = {
    'best_model_tuned': best_model_name,
    'holdout_accuracy': acc_t,
    'holdout_precision': prec_t,
    'holdout_recall': rec_t,
    'holdout_f1': f1_t,
    'holdout_roc_auc': roc_t,
    'tn': cm_t[0,0], 'fp': cm_t[0,1], 'fn': cm_t[1,0], 'tp': cm_t[1,1],
}
row.update({k: str(v) for k, v in best_params.items()})
pd.DataFrame([row]).to_csv('holdout_metrics_after_tuning.csv', index=False)

pd.read_csv('holdout_metrics_after_tuning.csv')

,best_model_tuned,holdout_accuracy,holdout_precision,holdout_recall,holdout_f1,holdout_roc_auc,tn,fp,fn,tp,clf__C,clf__penalty,clf__solver,pca__n_components
0,LogisticRegression,0.811348,0.684932,0.534759,0.600601,0.852311,472,46,87,100,3.0,l2,lbfgs,NaN


###12) Export preprocessed feature names for interpretation (decoded for human readable feature column)

In [57]:
preprocessor.fit(X_train)
num_feats = preprocessor.named_transformers_['num'].named_steps['scaler'].get_feature_names_out(num_cols)
cat_feats = preprocessor.named_transformers_['cat'].named_steps['onehot'].get_feature_names_out(cat_cols)
pd.DataFrame({'feature': np.concatenate([num_feats, cat_feats])}).to_csv('preprocessed_feature_names.csv', index=False)

print(pd.read_csv('preprocessed_feature_names.csv'))

                                    feature
0                             SeniorCitizen
1                                    tenure
2                            MonthlyCharges
3                              TotalCharges
4                             gender_Female
5                               gender_Male
6                                Partner_No
7                               Partner_Yes
8                             Dependents_No
9                            Dependents_Yes
10                          PhoneService_No
11                         PhoneService_Yes
12                         MultipleLines_No
13           MultipleLines_No phone service
14                        MultipleLines_Yes
15                      InternetService_DSL
16              InternetService_Fiber optic
17                       InternetService_No
18                        OnlineSecurity_No
19       OnlineSecurity_No internet service
20                       OnlineSecurity_Yes
21                          Onli